In [ ]:
import json
import glob
import os

import pandas as pd
from dotenv import load_dotenv
from together import Together
from tqdm import tqdm

load_dotenv()

DATA_ROOT = "../../data"

MODELS = {
    "qwen":        "Qwen/Qwen3-235B-A22B-Instruct-2507-tput",
    "gemma_small": "google/gemma-3n-E4B-it",
}

MODEL_KEY   = "gemma_small"   # qwen | gemma_small
PROMPT_TYPE = "full"   # full | reduced

MODEL_NAME  = MODELS[MODEL_KEY]
INPUT_DIR   = os.path.join(DATA_ROOT, "generation_input",    MODEL_KEY, PROMPT_TYPE)
RESULTS_DIR = os.path.join(DATA_ROOT, "generation_results_raw", MODEL_KEY, PROMPT_TYPE)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Model  : {MODEL_NAME}")
print(f"Prompt : {PROMPT_TYPE}")
print(f"Input  : {INPUT_DIR}")
print(f"Output : {RESULTS_DIR}")

## Get input files

In [ ]:
all_input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.jsonl")))

processed_channels = {
    os.path.basename(f).split("_")[-2]
    for f in glob.glob(os.path.join(RESULTS_DIR, "*"))
}
print(processed_channels)

data_files = [
    f for f in all_input_files
    if os.path.basename(f).split("_")[-2] not in processed_channels
]

print(f"Found {len(all_input_files)} input files."
      f" Found {len(processed_channels)} processed."
      f" Remaining: {len(data_files)}")
data_files

In [ ]:
all_input_ids = []
for file in tqdm(data_files):
    with open(file) as f:
        for line in f:
            all_input_ids.append(json.loads(line)["custom_id"])
print(f"Total input IDs: {len(all_input_ids)}")

## Upload files and submit batch jobs

In [ ]:
together_api_key = os.getenv("TOGETHER_API_KEY")
client = Together(api_key=together_api_key)

for file_path in data_files:
    print(file_path)
    file_resp = client.files.upload(file=file_path, purpose="batch-api", check=False)
    file_id   = file_resp.id
    batch     = client.batches.create(input_file_id=file_id, endpoint="/v1/chat/completions")
    print(batch.job.id)

## Load results

In [ ]:
DATE_FROM = "2026-05-01"  
DATE_TO   = "2026-05-15"

batches = client.batches.list()
all_batches_df = pd.DataFrame([batch.__dict__ for batch in batches])

run_batches_df = all_batches_df[
    (all_batches_df["x_model_id"] == MODEL_NAME) &
    (all_batches_df["created_at"] >= DATE_FROM) &
    (all_batches_df["created_at"] <  DATE_TO)
].sort_values("created_at", ascending=False)

print(f"Found {len(run_batches_df)} batches for {MODEL_KEY}/{PROMPT_TYPE}")
# run_batches_df

In [ ]:
all_outputs = []
for _, row in run_batches_df.iterrows():
    batch = client.batches.retrieve(row["id"])
    if batch.status == "COMPLETED":
        with client.files.with_streaming_response.content(
            id=row["output_file_id"]
        ) as response:
            with open("batch_output.jsonl", "wb") as f:
                for chunk in response.iter_bytes():
                    f.write(chunk)
    print(f"Batch_id: {row['id']}, status: {batch.status}, output_file_id: {row['output_file_id']}")
    with open("batch_output.jsonl") as f:
        all_outputs.extend(f.readlines())

print(f"Total outputs collected: {len(all_outputs)}")

In [ ]:
id_prefix = f"{MODEL_KEY}_{PROMPT_TYPE}_"
all_outputs = [line for line in all_outputs if json.loads(line)["custom_id"].startswith(id_prefix)]
print(f"Outputs after prefix filter ({id_prefix!r}): {len(all_outputs)}")

## Save

In [ ]:
all_output_ids = [json.loads(line)["custom_id"] for line in all_outputs]

missing_ids    = set(all_input_ids) - set(all_output_ids)
redundant_ids  = set(all_output_ids) - set(all_input_ids)

print(f"Missing   input_ids in outputs : {len(missing_ids)}")
print(f"Redundant output_ids           : {len(redundant_ids)}")

# Drop outputs that have no matching input
all_outputs = [line for line in all_outputs if json.loads(line)["custom_id"] not in redundant_ids]

output_file = os.path.join(RESULTS_DIR, "all_generation_results.jsonl")
with open(output_file, "w") as f:
    f.writelines(all_outputs)

print(f"Saved {len(all_outputs)} records → {output_file}")

In [ ]:
# from together import Together

# together_api_key = os.getenv("TOGETHER_API_KEY")
# client = Together(api_key=together_api_key)
# MODEL = "google/gemma-4-31B-it"
# ACTIVE = {"VALIDATING", "IN_PROGRESS"}

# for batch in client.batches.list():
#     print(batch)
#     if batch.x_model_id == MODEL and batch.status in ACTIVE:
#         client.batches.cancel(batch.id)
#         print(f"Cancelled {batch.id}")